In [1]:
import os
import json
import shutil
import random

import matplotlib.pyplot as plt

# ==================================================
# CONFIG
# ==================================================

BASE_DIR = (
    "/oscar/data/epavlick/aparasel/"
    "Binding-Problem-Brown/search-dataset/"
    "visual_search_dataset"
)

DISTRACTOR_COUNTS = [
    5,10,15,20,25,
    30,35,40,45,50
]

CONDITIONS = [
    "disjunctive",
    "conjunctive",
]

IMAGES_PER_BIN = 100

SEED = 42

random.seed(SEED)



def generate_search_trial(
    condition,
    target_present,
    num_distractors,
    save_path,
):

    fig, ax = plt.subplots(
        figsize=(6,6),
        dpi=100,
    )

    ax.set_xlim(0.5, 9.5)
    ax.set_ylim(0.5, 9.5)
    ax.axis("off")

    occupied = []

    metadata = {
        "condition": condition,
        "target_present": target_present,
        "num_distractors": num_distractors,
        "objects": [],
    }

    def get_random_position():

        for _ in range(1000):

            pos = (
                random.uniform(1.0,9.0),
                random.uniform(1.0,9.0),
            )

            threshold = (
                0.65
                if num_distractors <= 25
                else 0.45
            )

            if all(
                (
                    (pos[0]-p[0])**2
                    +
                    (pos[1]-p[1])**2
                ) > threshold
                for p in occupied
            ):
                return pos

        return (
            random.uniform(1.0,9.0),
            random.uniform(1.0,9.0),
        )


    if target_present:

        pos = get_random_position()
        occupied.append(pos)

        if condition == "disjunctive":

            shape = "circle"
            color = "green"

            ax.plot(
                pos[0],
                pos[1],
                marker="o",
                color=color,
                markersize=14,
                linestyle="None",
            )

        else:

            shape = "L"
            color = "green"

            ax.text(
                pos[0],
                pos[1],
                shape,
                color=color,
                fontsize=22,
                ha="center",
                va="center",
                fontweight="bold",
            )

        metadata["objects"].append({
            "role": "target",
            "shape": shape,
            "color": color,
            "position": [
                float(pos[0]),
                float(pos[1]),
            ],
        })

    # ==========================================
    # DISTRACTORS
    # ==========================================

    for _ in range(num_distractors):

        pos = get_random_position()
        occupied.append(pos)

        if condition == "disjunctive":

            shape = "circle"
            color = "red"

            ax.plot(
                pos[0],
                pos[1],
                marker="o",
                color=color,
                markersize=14,
                linestyle="None",
            )

        else:

            shape, color = random.choice([
                ("L","red"),
                ("T","green"),
            ])

            ax.text(
                pos[0],
                pos[1],
                shape,
                color=color,
                fontsize=22,
                ha="center",
                va="center",
                fontweight="bold",
            )

        metadata["objects"].append({
            "role": "distractor",
            "shape": shape,
            "color": color,
            "position": [
                float(pos[0]),
                float(pos[1]),
            ],
        })

    plt.savefig(
        save_path,
        bbox_inches="tight",
        pad_inches=0.1,
    )

    plt.close()

    json_path = save_path.replace(
        ".png",
        ".json",
    )

    with open(json_path, "w") as f:

        json.dump(
            metadata,
            f,
            indent=2,
        )

def generate_dataset():

    if os.path.exists(BASE_DIR):

        print("Deleting old dataset...")
        shutil.rmtree(BASE_DIR)

    os.makedirs(BASE_DIR)

    for condition in CONDITIONS:

        print(f"\nGenerating {condition}")

        folder = os.path.join(
            BASE_DIR,
            condition,
        )

        os.makedirs(folder)

        for nd in DISTRACTOR_COUNTS:

            trials = (
                [True]*50
                +
                [False]*50
            )

            random.shuffle(trials)

            for trial, target_present in enumerate(trials):

                fname = (
                    f"{condition}"
                    f"_distractors_{nd}"
                    f"_trial_{trial:03d}"
                    f"_target_{str(target_present).lower()}.png"
                )

                generate_search_trial(
                    condition=condition,
                    target_present=target_present,
                    num_distractors=nd,
                    save_path=os.path.join(
                        folder,
                        fname,
                    ),
                )
        print(
            f"Finished {condition}"
        )
    print("\nDone.")


if __name__ == "__main__":

    generate_dataset()

Deleting old dataset...

Generating disjunctive
Finished disjunctive

Generating conjunctive
Finished conjunctive

Done.


In [1]:
import os
from collections import Counter
from PIL import Image
import numpy as np

# ==================================================
# CONFIG
# ==================================================

ROOT = "/oscar/data/epavlick/aparasel/Binding-Problem-Brown/search-dataset/visual_search_dataset"

# ==================================================
# COLLECT IMAGE SIZES
# ==================================================

sizes = []
image_info = []

for dirpath, _, filenames in os.walk(ROOT):
    for filename in filenames:
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):

            path = os.path.join(dirpath, filename)

            try:
                with Image.open(path) as img:
                    w, h = img.size

                sizes.append((w, h))
                image_info.append((path, w, h))

            except Exception as e:
                print(f"Failed to read {path}: {e}")

# ==================================================
# SUMMARY
# ==================================================

if len(sizes) == 0:
    print("No images found.")
    raise SystemExit

widths = np.array([w for w, h in sizes])
heights = np.array([h for w, h in sizes])

print("=" * 60)
print(f"Images checked: {len(sizes)}")
print("=" * 60)

print(f"Average width : {widths.mean():.2f}")
print(f"Average height: {heights.mean():.2f}")

print(f"Median width : {np.median(widths)}")
print(f"Median height: {np.median(heights)}")

print(f"Min size: {widths.min()} x {heights.min()}")
print(f"Max size: {widths.max()} x {heights.max()}")

# ==================================================
# MOST COMMON SIZE
# ==================================================

size_counts = Counter(sizes)
median_size = size_counts.most_common(1)[0][0]

print("\nMost common image size:", median_size)
print("Count:", size_counts[median_size])

# ==================================================
# FIND IMAGES WITH DIFFERENT SIZE
# ==================================================

different = [
    (path, w, h)
    for path, w, h in image_info
    if (w, h) != median_size
]

print("\nImages different from the most common size:", len(different))

if different:
    print("\nExamples:")
    for path, w, h in different[:20]:
        print(f"{w}x{h}  {path}")

    if len(different) > 20:
        print(f"... and {len(different)-20} more.")
else:
    print("All images have identical dimensions.")

Images checked: 2000
Average width : 484.00
Average height: 482.00
Median width : 484.0
Median height: 482.0
Min size: 484 x 482
Max size: 484 x 482

Most common image size: (484, 482)
Count: 2000

Images different from the most common size: 0
All images have identical dimensions.
